# Project 8: Image Processing

In this project, you will be using Python to load, edit, and write image files. Specifically, we will be applying _filters_ to images by modifying their pixel values.

The image file format we will use in this project is [PPM](http://netpbm.sourceforge.net/doc/ppm.html). PPM is an interesting picture format because it stores the RGB values for each pixel as numbers in a text file. For example, the following is a PPM image:

```
P3
2 2
255
0   0   255    255 0   0
255 0   0      0   0   255
```

This PPM image is a 2x2 pixel square, where the corners are alternating red and blue. In fact, if we wrote it to a file and displayed it, that's what we would see!

<small>Of course, you will have to zoom way in in order to be able to see just 4 pixels...</small>

In [ ]:
from cs1.ppm import display_ppm # the cs1.ppm library has a function to display ppm images.

# Write our simple PPM to a file, one line at a time.
# (You'll learn how writing to a file works in Step 6.)
outfile = open("simple.ppm", "w")
outfile.write("P3\n")
outfile.write("2 2\n")
outfile.write("255\n")
outfile.write("0 0 255 255 0 0\n")
outfile.write("255 0 0 0 0 255\n")
outfile.close()

# Display the image file.
display_ppm("simple.ppm")

## The PPM Image Format

PPM encodes an image in a human-readable plain text file. This file is divided into two parts:

* A __header__ that describes the image size and how color values are represented.
* A __body__ that encodes the RGB values for each pixel in the image.

Let's use a (slightly) larger example to illustrate these:

```
P3
4 4
255
255 0   0       0   255 0      0   0   255    255 255 255
0   0   0       255 0   0      0   255 0      0   0   255
255 0   255     0   0   0      255 0   0      0   255 0
0   255 255     255 0   255    0   0   0      255 0   0
```


### The PPM Header

The PPM header is the first three lines in the file.

```
P3
4 4
255
```

* The first line (`P3`) specifies the file encoding--which _type_ of PPM image it is. There are several, but `P3` is the one that we will be using in this project. So each file will always start with `P3`.
* The second line specifies the width (columns) and height (rows) of the image. Our first image was a 2x2 image; this one is 4x4.
* The last line indicates the maximum value for red, green, and blue values. We'll always use a max value of 255 (as we do in our cs1 graphics library).

### The PPM Body

Below the header is the body. In the body, the color values for each pixel are listed.

```
255 0   0       0   255 0      0   0   255    255 255 255
0   0   0       255 0   0      0   255 0      0   0   255
255 0   255     0   0   0      255 0   0      0   255 0
0   255 255     255 0   255    0   0   0      255 0   0
```

(We added extra spaces here to line up the pixels. In the real files, there's exactly one space between values.)

Each line in the file represents one row in the image. On a line, each pixel is represented by three values between 0 and 255. So, the first pixel on row one in this image has the RGB value $(255, 0, 0)$. The second pixel has the value $(0, 255, 0)$.

If we enlarged this image, it would look like this:

<img src="https://storage.googleapis.com/141-files/fourbyfour.png" width=200/>

Of course, it is much smaller.

In [ ]:
from cs1.ppm import display_ppm

display_ppm("test.ppm")

There are a few PPM images in this folder. They are `lynx.ppm`, `rhodes.ppm`, `bclc.ppm`, `blobby.ppm`, `hounddogs.ppm`, and `joly.ppm`. The first 3 images are related to Rhodes. The last 3 images are some of the CS professors' pets.

__Note:__ PPM is an inefficient format--images stored as PPM are orders of magnitude larger than images stored as JPG, for example. That's why these files are so big!

In [ ]:
from cs1.ppm import display_ppm

display_ppm("lynx.ppm")


# uncomment/comment the following to see other test images!

#display_ppm("rhodes.ppm")
#display_ppm("bclc.ppm")
#display_ppm("blobby.ppm")    # Dr. Kirlin claims he is lame for having no pets,
                              # but has supplied us with his snowman: Blobby McPastaface
#display_ppm("hounddogs.ppm") # these are Dr. Sanders's dachshunds
#display_ppm("joly.ppm")      # this is Dr. Superdock's dog Joly

<hr>

# How your program will work

You will write your program in `imgfilter.py`. Your program should begin by asking the user for the name of a PPM file. It will then show a menu of filters and ask the user which one to apply. You'll implement the following filters:

1. Negate red
2. Negate green
3. Negate blue
4. Negate all
5. Remove red
6. Remove green
7. Remove blue
8. Flip horizontally
9. Grayscale

Your program will then write a new PPM file with the filter applied, and display the new image. The output file name should include the name of the filter. Here's a sample run of the finished program:

```
PPM file? lynx.ppm
Filters:
1. Negate red
2. Negate green
3. Negate blue
4. Negate all
5. Remove red
6. Remove green
7. Remove blue
8. Flip horizontally
9. Grayscale
Select a filter: 12
Bad input, try again!
Select a filter: 9
Output file with filter 9 applied is named lynx_grayscale.ppm
```

After that, your program displays the new image. You can see what each filter should look like at the end of Step 6.

We will build this program incrementally, by starting with small, simple functions, and combining them to create the full program:

1. Look over the starter code
2. Write `make_output_filename`
3. Write the filter functions: `negate`, `flatten`, `flip`, and `grayscale`
4. Write `get_channel` and `make_line`
5. Write `process_list`
6. Finish `main()`: an input validation loop, and the loop that reads and writes the files

Test each step before you move on to the next one!

<hr>

## Step 1: Look over the starter code

Open `imgfilter.py`. First, fill in the `Author`, `Pledge`, and `Description` fields at the top of the file.

Then read over the rest of the file. It has a comment for each function you'll write, and a `main` function that is partly written for you. The program will come together by writing a series of functions.

__Make sure you read the comments in the functions!__ The comments above each function tell you __what__ it's supposed to do. Some functions also have comments inside that tell you __how__ to do it.

<hr>

## Step 2: Write `make_output_filename`

Write the function `make_output_filename`. It makes the name of the output file, based on the original file name and the filter that was applied. For example, if the user applies filter 9 (Grayscale) to `rhodes.ppm`, the output file should be named `rhodes_grayscale.ppm`.

Test your function in the cells below. __You should also write some tests of your own.__

In [ ]:
# This imports all the functions in your file so that they can be used in the notebook.
from cs1.notebooks import *
reload_functions("imgfilter.py")

print(make_output_filename("rhodes.ppm", 9)) # should print rhodes_grayscale.ppm
print(make_output_filename("lynx.ppm", 1)) # should print lynx_negate_red.ppm

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Write at least two more tests here for make_output_filename.


In [ ]:
# Runs a set of tests against make_output_filename.
from cs1.notebooks import *
ok_runtests("p8.ok", "q1")

<hr>

## Step 3: Write the filter functions

Next, write the functions `negate`, `flatten`, `flip`, and `grayscale`. Each one takes a list of numbers (or three lists, for `grayscale`) and returns a new list. Test each function in the cells below, and write some tests of your own.

In this project, a *filter* means an image filter, like the ones on Instagram. That's different from filtering a list, where you keep only some of the items! These functions don't skip any values, so each new list is the same length as the original.

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

print(negate([1, 2, 3, 200, 100, 150])) # should print [254, 253, 252, 55, 155, 105]

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Write at least two more tests here for negate.


In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

print(flatten([1, 2, 3, 200, 100, 150])) # should print [0, 0, 0, 0, 0, 0]

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Write at least two more tests here for flatten.


In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

print(flip([1, 2, 3, 200, 100, 150])) # should print [150, 100, 200, 3, 2, 1]

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Write at least two more tests here for flip.


In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

print(grayscale([1, 200], [2, 100], [3, 150])) # should print [2, 150]

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Write at least two more tests here for grayscale.


In [ ]:
# Runs a set of tests against your filters.
from cs1.notebooks import *
ok_runtests("p8.ok", "q2")

<hr>

## Step 4: Write `get_channel` and `make_line`

Each line of a PPM file after the header is one row of pixels. When your program reads a line, it will split it into a list of strings. For example, the line `255 0 10 0 128 20` becomes `["255", "0", "10", "0", "128", "20"]`. That's 2 pixels: (255, 0, 10) and (0, 128, 20).

Write these two functions:

* `get_channel` pulls out the values for one color (one *channel*) from that list, as integers.
* `make_line` turns a list of numbers back into a string, so it can be written to a file.

Test them in the cells below, and write some tests of your own.

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

values = ["255", "0", "10", "0", "128", "20"]
print(get_channel(values, 0)) # should print [255, 0]
print(get_channel(values, 1)) # should print [0, 128]
print(get_channel(values, 2)) # should print [10, 20]

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Write at least two more tests here for get_channel.


In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Jupyter shows the string in quotes, so you can see the space at the end.
make_line([0, 0, 10, 255, 128, 20]) # should show '0 0 10 255 128 20 '

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Write at least two more tests here for make_line.


In [ ]:
# Runs a set of tests against get_channel and make_line.
from cs1.notebooks import *
ok_runtests("p8.ok", "q3")

<hr>

## Step 5: Write `process_list`

Now write `process_list`. It puts the pieces together: it takes one row of pixels (a list of strings, like the one above), applies a filter to it, and returns the new row as a list of numbers. It should call `get_channel` and your filter functions. Follow the algorithm in the comments in `process_list`, then test it in the cells below and write some tests of your own.

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

values = ["255", "0", "10", "0", "128", "20"]
print(process_list(values, 1)) # negate red: should print [0, 0, 10, 255, 128, 20]
print(process_list(values, 8)) # flip horizontally: should print [0, 128, 20, 255, 0, 10]
print(process_list(values, 9)) # grayscale: should print [88, 88, 88, 49, 49, 49]

In [ ]:
from cs1.notebooks import *
reload_functions("imgfilter.py")

# Write at least two more tests here for process_list.


In [ ]:
# Runs a set of tests against process_list.
from cs1.notebooks import *
ok_runtests("p8.ok", "q4")

<hr>

## Step 6: Finish `main()`

`main()` is partly written for you. You need to add two loops (the comments in `main()` tell you where they go):

1. An input validation loop to ensure that the user enters a filter number from 1 to 9.
2. The loop that reads the input file one line at a time and writes the output file. It should use `process_list` and `make_line`.

The second loop needs to write to a file. Here's how that works.

### Writing to a file

`main()` already opens the input file for reading and the output file for writing (with `"w"`), and closes both of them at the end. You'll write the loop in between. You already know how to read a file one line at a time. To write to a file, use the `write` function:

```python
output_file.write("P3\n")
```

* `write` works like `print`, except what you write __must__ be a string. If you want to write numbers, convert them to strings with `str` first.
* You can only write one thing at a time (no commas like `print`). Combine pieces into one string with `+` instead.

#### What's that `\n`?

`\n` is the __newline character__. It's an invisible character that marks the end of a line in a file.

Remember how we used `line = line.rstrip()` when reading from a file? That's because every line you read from a file ends with a `\n`, and `rstrip` removes it.

Writing works the other way around. `print` automatically adds a `\n` to the end of everything it prints (that's why each `print` starts a new line), but `write` doesn't. So this code:

```python
output_file.write("Batman")
output_file.write("Wonder Woman")
```

puts `BatmanWonder Woman` on one line in the file. If you want each piece on its own line, you have to add the `\n` yourself:

```python
output_file.write("Batman\n")
output_file.write("Wonder Woman\n")
```

So in this project, `rstrip` every line you read, and add `"\n"` back to every line you write.

### Running your program

To test `main()`, you'll need to run the entire program. You can run your full program using the cell below. Be sure to test your input validation loop too.

If `display_ppm` gives you an error (even if the image shows up first), there's a problem with the file your program wrote. Try `test.ppm`, and use the check below to see exactly what your program wrote.

In [ ]:
# Run your full program. I suggest using the smaller images (test.ppm and lynx.ppm) to start with.
%run imgfilter.py

#### Check your work on `test.ppm`

Before you try the big images, run your program (using the cell above) on `test.ppm` with filter 1 (Negate red), then run the cell below to see what your program wrote. It should be exactly this:

```
P3
4 4
255
0 0 0 255 255 0 255 0 255 0 255 255
255 0 0 0 0 0 255 255 0 255 0 255
0 0 255 255 0 0 0 0 0 255 255 0
255 255 255 0 0 255 255 0 0 0 0 0
```

Only the red values (the 1st, 4th, 7th, and 10th numbers on each row) should have changed. Compare this to the 4x4 example in The PPM Body section above to see the difference.

In [ ]:
# Run this after you run your program on test.ppm with filter 1 (Negate red).
file = open("test_negate_red.ppm", "r")
for line in file:
    print(line.rstrip())
file.close()

#### What your output should look like

Here's a reference output for the `bclc.ppm` image.

<table>
    <tr>
        <td><img src="https://storage.googleapis.com/141-files/bclc_negate_red.jpg" width=200/><br>negate red
        <td><img src="https://storage.googleapis.com/141-files/bclc_negate_green.jpg" width=200/><br>negate green
        <td><img src="https://storage.googleapis.com/141-files/bclc_negate_blue.jpg" width=200/><br>negate blue
    </tr>
    <tr>
        <td><img src="https://storage.googleapis.com/141-files/bclc_remove_red.jpg" width=200/><br>remove red
        <td><img src="https://storage.googleapis.com/141-files/bclc_remove_green.jpg" width=200/><br>remove green
        <td><img src="https://storage.googleapis.com/141-files/bclc_remove_blue.jpg" width=200/><br>remove blue
    </tr>
    <tr>
        <td><img src="https://storage.googleapis.com/141-files/bclc_negate_all.jpg" width=200/><br>negate all
        <td><img src="https://storage.googleapis.com/141-files/bclc_flip_horizontally.jpg" width=200/><br>flip horizontally
        <td><img src="https://storage.googleapis.com/141-files/bclc_grayscale.jpg" width=200/><br>grayscale
    </tr>
</table>

<hr>

### Grading

As a reminder, you will be graded on the following criteria:

* Whether your program and individual functions work. Your program should work for any PPM image and any of the 9 filters.
* Whether you included appropriate tests for each function as directed in the notebook.
* Whether your program is fully commented, including the header at the top of the program, comments before each function, and appropriate comments elsewhere in the program.
* Whether your program follows generally good programming practices and style.

## Feedback Questions

__In the cell below, please answer the following questions about the program.__
1. What did you struggle with the most?
2. What did you enjoy most about writing the program?
3. Please estimate about how long you spent working on this assignment.
4. Any other questions, concerns, comments about the program or material we have covered so far?

__Put your answers here!__ Double click this cell to edit and remove this text.

Feedback 1:

Feedback 2:

Feedback 3:

Feedback 4:

### Submitting

Once you have answered the four feedback questions above, run the following cell to submit. You can submit as many times as you like. Make sure that you click the link returned to you by okpy to verify that your notebook and program were both uploaded correctly.

In [ ]:
# Run this cell to submit.
# Submit as frequently as you like; I will only grade the last submission.
from cs1.notebooks import *
ok_submit("p8.ok")

## Additional Challenges

Try implementing different filters! Here are some ideas, roughly from easiest to hardest:

1. __Warm or cool:__ Many Instagram filters are easily duplicated by just "warming" or "cooling" an image. To warm an image, reduce the blue values (for example, multiply each one by 0.8 and convert back to an integer). To cool an image, reduce the red values instead.
2. __Brighten:__ Add 50 to every value. Values can't go above 255, so anything bigger than 255 should become 255.
3. __Black and white:__ Average each pixel's red, green, and blue values (like grayscale). If the average is more than 127, make the pixel white (255, 255, 255). Otherwise, make it black (0, 0, 0).
4. __Sepia:__ Give the image an old-photo look. For each pixel, new red = 0.393 × red + 0.769 × green + 0.189 × blue, new green = 0.349 × red + 0.686 × green + 0.168 × blue, and new blue = 0.272 × red + 0.534 × green + 0.131 × blue. Convert each one to an integer, and make anything bigger than 255 become 255.
5. __Flip vertically:__ Turn the image upside down. This one is harder, because you have to read all of the rows (and store them in a list) before you can write any of them.

If you add a new filter, also add it to the menu in `main()`, to the `mods` list in `make_output_filename`, to `process_list`, and to your input validation loop. Please include comments to make it easy for the graders to see what additional challenges you completed.

Remember, you can submit as many times as you like!